# August 2026 verification · **03 — Silver: Cancer Waiting Times**

This is the same treatment my existing CWT pipeline uses, applied to the
2025-26 Final and 2026-27 (April-August, Provisional) Combined CSVs:

* **Provider** basis only. Commissioner rows count the same patients from the other side.
* The England `Org_Code == "Total"` rows are **taken out** of the provider rows and kept separately, so I can prove the trusts add up to them.
* Compliance is recomputed as `Within / Total`, never taken from `Performance`.
* Suppressed values stay null. A suppressed cell never becomes a zero.

Headline slices, matching the statistical release's definitions:

| measure | `Standard_or_Item` | `Cancer_Type` | `Referral_Route_or_Stage` | `Treatment_Modality` |
|---|---|---|---|---|
| 28-day Faster Diagnosis | `FDS` | `ALL CANCERS` | `ALL ROUTES` | (blank) |
| 31-day Combined | `31D` | `ALL CANCERS` | `ALL STAGES` | `ALL MODALITIES` |
| 62-day Combined | `62D` | `ALL CANCERS` | `ALL ROUTES` | `ALL MODALITIES` |

31-day isn't one of the claims under test. I carry it because it costs
nothing, and the release quotes it too.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import BRONZE_DIR, SILVER_DIR  # noqa: E402

OUT = SILVER_DIR / "cwt"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 220)

FILES = {
    "2025-26 final": BRONZE_DIR / "nhse_cwt_2025-26" / "2025-26-Apr-Mar-Cumulative-Monthly-Combined-CSV-Final.csv",
    "2026-27 provisional": BRONZE_DIR / "nhse_cwt_2026-08" / "2026-27-Apr-Aug-Cumulative-Monthly-Combined-CSV-Provisional.csv",
}
SINGLE_MONTH = BRONZE_DIR / "nhse_cwt_2026-08" / "August-2026-Monthly-Combined-CSV-Provisional.csv"

HEADLINE = {
    "FDS": dict(Standard_or_Item="FDS", Cancer_Type="ALL CANCERS", Referral_Route_or_Stage="ALL ROUTES", Treatment_Modality=None),
    "31D": dict(Standard_or_Item="31D", Cancer_Type="ALL CANCERS", Referral_Route_or_Stage="ALL STAGES", Treatment_Modality="ALL MODALITIES"),
    "62D": dict(Standard_or_Item="62D", Cancer_Type="ALL CANCERS", Referral_Route_or_Stage="ALL ROUTES", Treatment_Modality="ALL MODALITIES"),
}

## Step 1 — Load, and parse the period honestly

The two files write dates differently: 2025-26 uses `01/08/2025`, 2026-27 uses
`2026-08-01`. I learned in my CWT pipeline that string comparison silently
drops one of them. So I parse each file against its own explicit format and
fail if a single value doesn't parse.

In [2]:
def parse_period(s: pd.Series) -> pd.Series:
    for fmt in ("%Y-%m-%d", "%d/%m/%Y"):
        out = pd.to_datetime(s, format=fmt, errors="coerce")
        if out.notna().all():
            return out
    raise ValueError(f"unparseable Period values: {s[out.isna()].unique()[:5]}")

frames = []
for label, path in FILES.items():
    d = pd.read_csv(path, low_memory=False)
    d["period_date"] = parse_period(d["Period"])
    d["vintage"] = label.split()[1]
    frames.append(d)
    print(f"{label}: {len(d):,} rows, {d.period_date.min():%b %Y} to {d.period_date.max():%b %Y}")
raw = pd.concat(frames, ignore_index=True)
raw["period"] = raw.period_date.dt.strftime("%Y-%m")
assert not raw.duplicated(["period", "Basis", "Org_Code", "Standard_or_Item", "Cancer_Type",
                           "Referral_Route_or_Stage", "Treatment_Modality"]).any(), "grain violated in the raw files"
expected = pd.period_range("2025-04", "2026-08", freq="M").strftime("%Y-%m").tolist()
assert sorted(raw.period.unique()) == expected, "CWT months are not the unbroken April 2025 - August 2026 run"

2025-26 final: 325,056 rows, Apr 2025 to Mar 2026
2026-27 provisional: 129,681 rows, Apr 2026 to Aug 2026


## Step 2 — Headline slices, provider basis, England row held apart

In [3]:
def headline_slice(d: pd.DataFrame, spec: dict) -> pd.DataFrame:
    m = d.Basis.eq("Provider")
    for col, val in spec.items():
        m &= d[col].isna() if val is None else d[col].eq(val)
    return d[m]

rows = []
for measure, spec in HEADLINE.items():
    s = headline_slice(raw, spec)
    s = s.assign(measure=measure)
    rows.append(s)
hl = pd.concat(rows, ignore_index=True)

provider = hl[hl.Org_Code != "Total"].copy()
england_row = hl[hl.Org_Code == "Total"].copy()
# One England row per measure per month, no more and no less.
assert england_row.groupby(["measure", "period"]).size().eq(1).all()
assert len(england_row) == len(HEADLINE) * len(expected)
assert not provider.duplicated(["measure", "period", "Org_Code"]).any()

# I keep the waiting-time bands too, so Gold can say how late, not just whether
# late. The bands are reused physically across standards, so each measure gets
# only the columns NHS England populates for it, and I check they add up.
BANDS = {
    "FDS": ["Within_14_days", "In_15_to_28_days", "In_29_to_42_days", "In_43_to_62_days", "After_62_days"],
    # 31-day stops at 62 days: its last band is After_62_days.
    "31D": ["Within_31_days", "In_32_to_38_days", "In_39_to_48_days", "In_49_to_62_days", "After_62_days"],
    # For 62-day, After_62_days just repeats After; the 63-104+ bands split it.
    "62D": ["Within_31_days", "In_32_to_38_days", "In_39_to_48_days", "In_49_to_62_days", "In_63_to_76_days",
            "In_77_to_90_days", "In_91_to_104_days", "After_104_days"],
}
ALL_BANDS = sorted({b for v in BANDS.values() for b in v})
for m, cols in BANDS.items():
    x = hl[hl.measure == m]
    gap = (x[cols].sum(axis=1, min_count=1) - x.Total).abs()
    assert (gap.dropna() < 1e-6).all(), f"{m}: bands do not sum to Total on {int((gap >= 1e-6).sum())} rows"
keep = ["measure", "period", "vintage", "Org_Code", "Parent_Org", "Org_Name", "Total", "Within", "After"] + ALL_BANDS
cwt_provider = provider[keep].rename(columns={"Org_Code": "org_code", "Parent_Org": "parent_org", "Org_Name": "org_name",
                                              "Total": "total", "Within": "within", "After": "after"})
cwt_england_published = england_row[keep + ["Performance"]].rename(columns={
    "Org_Code": "org_code", "Parent_Org": "parent_org", "Org_Name": "org_name",
    "Total": "total", "Within": "within", "After": "after", "Performance": "performance_published"})

## Step 3 — Do the trusts add up to England, inside the file?

This is not yet the gate. The gate compares with the release and the national
time series. This only checks that the file agrees with itself.

In [4]:
summed = cwt_provider.groupby(["measure", "period"])[["total", "within", "after"]].sum()
eng = cwt_england_published.set_index(["measure", "period"])[["total", "within", "after"]]
diff = (summed - eng)
print("largest |provider sum − England row| by column:", diff.abs().max().to_dict())
assert (diff.abs() < 1e-6).all().all(), diff[(diff.abs() >= 1e-6).any(axis=1)]
assert (summed.within + summed.after - summed.total).abs().max() < 1e-6, "Within + After != Total"

largest |provider sum − England row| by column: {'total': 0.0, 'within': 0.0, 'after': 0.0}


## Step 4 — The single-month August file must agree with the cumulative one

NHS England publishes August twice: on its own, and inside April-August. If
the two disagree, I want to know before I pick one.

In [5]:
single = pd.read_csv(SINGLE_MONTH, low_memory=False)
cols = ["Basis", "Org_Code", "Standard_or_Item", "Cancer_Type", "Referral_Route_or_Stage", "Treatment_Modality"]
cum_aug = raw[raw.period == "2026-08"]
mm = single.merge(cum_aug, on=cols, how="outer", suffixes=("_single", "_cum"), indicator=True)
assert (mm._merge == "both").all(), mm._merge.value_counts()
for c in ["Total", "Within", "After"]:
    a, b = mm[f"{c}_single"], mm[f"{c}_cum"]
    assert ((a == b) | (a.isna() & b.isna())).all(), f"single-month and cumulative August files disagree on {c}"
print(f"single-month August file = cumulative file, all {len(mm):,} rows")

single-month August file = cumulative file, all 25,767 rows


In [6]:
cwt_provider["compliance"] = cwt_provider.within / cwt_provider.total
cwt_provider.to_parquet(OUT / "cwt_provider_headline.parquet", index=False)
cwt_england_published.to_parquet(OUT / "cwt_england_row.parquet", index=False)
print(cwt_provider.groupby(["measure", "period"]).agg(trusts=("org_code", "nunique"), total=("total", "sum"),
                                                     within=("within", "sum")).assign(
    pct=lambda x: (100 * x.within / x.total).round(1)).loc[(slice(None), ["2025-08", "2026-07", "2026-08"]), :])

                 trusts     total    within   pct
measure period                                   
31D     2025-08     140   54415.0   49849.0  91.6
        2026-07     139   62098.0   57442.0  92.5
        2026-08     139   55049.0   50701.0  92.1
62D     2025-08     149   27533.0   19090.0  69.3
        2026-07     146   31519.0   22428.0  71.2
        2026-08     143   27909.0   20066.0  71.9
FDS     2025-08     142  273040.0  203750.0  74.6
        2026-07     140  308370.0  244594.0  79.3
        2026-08     140  273664.0  214991.0  78.6


## Step 5 — August 2025 as first published (provisional)

For a like-for-like year-on-year comparison I want August 2025 **provisional**
alongside August 2026 provisional. The 2025-26 Final file holds only the
revised figures. August 2025 came out in NHS England's pre-November-2025
layout, one CSV per standard with header rows on top, so I parse it here,
once, into the same columns as above.

In [7]:
OLD = BRONZE_DIR / "nhse_cwt_2025-08_provisional"
OLD_FILES = {
    "FDS": (OLD / "2.-28-Day-Faster-Diagnosis-All-Routes-Provider-Data.csv",
            {"TOTAL": "total", "WITHIN 28 DAYS": "within", "AFTER 28 DAYS": "after"}),
    "62D": (OLD / "7.-62-Day-Combined-All-Cancers-Provider-Data.csv",
            {"TOTAL": "total", "WITHIN 62 DAYS": "within", "AFTER 62 DAYS": "after"}),
}

def parse_old(path: Path, cols: dict) -> pd.DataFrame:
    raw_old = pd.read_csv(path, header=None, low_memory=False)
    hdr_row = raw_old.index[raw_old.iloc[:, 1].astype(str).str.startswith("ODS CODE")][0]
    d = raw_old.iloc[hdr_row + 1:].copy()
    d.columns = [str(c).strip() for c in raw_old.iloc[hdr_row]]
    # The 62-day file stacks five parts (all routes, then each route) with repeated
    # header rows. I keep Part A only: the rows whose route is ALL ROUTES.
    d = d[d["REFERRAL ROUTE"].eq("ALL ROUTES")]
    d = d.rename(columns={"ODS CODE (1)": "org_code", "ACCOUNTABLE PROVIDER": "org_name", **cols})
    for c in cols.values():
        d[c] = pd.to_numeric(d[c])
    return d[["org_code", "org_name", "total", "within", "after"]]

prov25 = []
for m, (path, cols) in OLD_FILES.items():
    assert pd.read_csv(path, header=None, nrows=2).iloc[1, 0] == "Aug-25", f"{path.name} is not August 2025"
    d = parse_old(path, cols).assign(measure=m, period="2025-08", vintage="provisional (first published)")
    eng = d[d.org_name.eq("ALL ENGLISH PROVIDERS")]
    trusts = d[d.org_code.notna() & ~d.org_name.eq("ALL ENGLISH PROVIDERS")]
    assert len(eng) == 1 and trusts.org_code.is_unique
    for c in ["total", "within", "after"]:
        assert abs(trusts[c].sum() - eng[c].iloc[0]) < 1e-6, f"{m} Aug-25 provisional: trusts do not sum to England on {c}"
    prov25.append(d)
    print(f"{m} Aug 2025 provisional: {len(trusts)} trusts sum exactly to England "
          f"{eng.total.iloc[0]:,.0f} / {eng.within.iloc[0]:,.0f} ({100 * eng.within.iloc[0] / eng.total.iloc[0]:.1f}%)")
pd.concat(prov25, ignore_index=True).to_parquet(OUT / "cwt_2025-08_provisional.parquet", index=False)

FDS Aug 2025 provisional: 140 trusts sum exactly to England 267,937 / 199,914 (74.6%)
62D Aug 2025 provisional: 148 trusts sum exactly to England 26,906 / 18,596 (69.1%)
